In [6]:
!pip install streamlit pandas numpy scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.8 MB/s eta 0:00:00


In [7]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# 1. ESENCIAL: Definir la función de preprocesamiento para que joblib la encuentre al cargar el pipeline
def arreglar_pdays(X):
    X_nuevo = X.copy()
    if 'pdays' in X_nuevo.columns:
        X_nuevo['pdays_contacted'] = (X_nuevo['pdays'] != -1).astype(int)
        X_nuevo['pdays'] = X_nuevo['pdays'].replace(-1, np.nan)
    return X_nuevo

# 2. Configuración de la página
st.set_page_config(page_title="Despliegue del Modelo - Banco", page_icon="🏦", layout="centered")
st.title("Predicción de Subscripción a un Depósito Bancario")
st.write("Esta aplicación utiliza el pipeline de `scikit-learn` cargado desde `modelo_final.joblib`.")

# 3. Cargar el modelo final
@st.cache_resource
def load_model():
    # Asegúrate de que el archivo se llame exactamente así como pide el PDF
    return joblib.load("modelo_final.joblib")

try:
    pipeline = load_model()
except Exception as e:
    st.error(f"Error al cargar el modelo: {e}")
    st.stop()

st.markdown("### Introduce los valores del cliente:")

# 4. Formulario de entradas basado en el Apéndice del PDF
with st.form("prediction_form"):

    st.subheader("Variables Numéricas")
    num_col1, num_col2 = st.columns(2)

    with num_col1:
        age = st.number_input("Edad (age)", min_value=18, max_value=120, value=30, step=1)
        balance = st.number_input("Balance anual medio (balance)", value=0)
        duration = st.number_input("Duración del último contacto (duration) [segundos]", min_value=0, value=150)
        campaign = st.number_input("Contactos en esta campaña (campaign)", min_value=1, value=1)

    with num_col2:
        pdays = st.number_input("Días desde el último contacto (pdays) [-1 = sin contacto]", min_value=-1, value=-1)
        previous = st.number_input("Contactos previos a esta campaña (previous)", min_value=0, value=0)

        # En caso de que tu dataset use 'day' (como día del mes) en vez de un día categórico:
        day = st.number_input("Día del último contacto (day)", min_value=1, max_value=31, value=15)

    st.subheader("Variables Categóricas")
    cat_col1, cat_col2 = st.columns(2)

    with cat_col1:
        job = st.selectbox("Tipo de trabajo (job)", ["admin.", "blue-collar", "entrepreneur", "housemaid", "management", "retired", "self-employed", "services", "student", "technician", "unemployed", "unknown"])
        marital = st.selectbox("Estado marital (marital)", ["divorced", "married", "single", "unknown"])
        education = st.selectbox("Nivel de educación (education)", ["primary", "secondary", "tertiary", "unknown", "basic.4y", "basic.6y", "basic.9y", "high.school", "illiterate", "professional.course", "university.degree"])
        default = st.selectbox("¿Algún crédito no devuelto? (default)", ["no", "yes", "unknown"])
        housing = st.selectbox("¿Tiene una hipoteca? (housing)", ["no", "yes", "unknown"])

    with cat_col2:
        loan = st.selectbox("¿Tiene un préstamo personal? (loan)", ["no", "yes", "unknown"])
        contact = st.selectbox("Tipo de contacto (contact)", ["cellular", "telephone", "unknown"])
        month = st.selectbox("Último mes de contacto (month)", ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])
        poutcome = st.selectbox("Resultado campaña anterior (poutcome)", ["failure", "nonexistent", "success", "unknown", "other"])
        # Si tu dataset utiliza day_of_week como categórica, descomenta la siguiente línea y ponla en los inputs
        # day_of_week = st.selectbox("Último día de contacto (day_of_week)", ["mon", "tue", "wed", "thu", "fri", "unknown"])

    st.markdown("---")
    submitted = st.form_submit_button("Realizar Predicción", use_container_width=True)

if submitted:
    # 5. Convertimos las entradas en un DataFrame de una única fila
    inputs = {
        "age": age,
        "job": job,
        "marital": marital,
        "education": education,
        "default": default,
        "balance": balance,
        "housing": housing,
        "loan": loan,
        "contact": contact,
        "day": day, # NOTA: Cambia esto a 'day_of_week': day_of_week si tu dataset emplea el día como categórico
        "month": month,
        "duration": duration,
        "campaign": campaign,
        "pdays": pdays,
        "previous": previous,
        "poutcome": poutcome
    }

    X_new = pd.DataFrame([inputs])

    try:
        # 6. Predecimos usando el Pipeline
        y_pred = pipeline.predict(X_new)[0]

        # Muestra la predicción principal
        resultado_texto = "SUSCRIBE el depósito ✅" if y_pred in ['yes', 1] else "NO SUSCRIBE el depósito ❌"
        st.success(f"### Resultado Predictivo: **{resultado_texto}** (Clase: {y_pred})")

        # Mostramos las probabilidades si el modelo lo permite
        if hasattr(pipeline, "predict_proba"):
            st.markdown("#### Probabilidades de la Predicción:")
            proba = pipeline.predict_proba(X_new)[0]

            # Intentar extraer los nombres de las clases (si están disponibles en el estimador)
            clases = pipeline.classes_ if hasattr(pipeline, "classes_") else [f"Clase {i}" for i in range(len(proba))]

            cols = st.columns(len(clases))
            for i, (cls, p) in enumerate(zip(clases, proba)):
                cols[i].metric(label=f"Probabilidad de {cls}", value=f"{p*100:.1f}%")

    except Exception as e:
        st.error(f"Error durante la predicción: {e}\n\n*Nota: Verifica si los nombres de las columnas enviadas coinciden exactamente con los que usaste durante el entrenamiento del modelo.*")

2026-03-26 15:35:22.285 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 15:35:22.286 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 15:35:22.474 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-26 15:35:22.476 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 15:35:22.478 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 15:35:22.481 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 15:35:22.485 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [8]:
import urllib
print("Copia esta IP. Te la pedirá Localtunnel en el siguiente paso:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Copia esta IP. Te la pedirá Localtunnel en el siguiente paso:
34.125.158.237


In [ ]:
!streamlit run mystreamlit.py &>/content/logs.txt &
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://six-tips-think.loca.lt
